# GARCH Model Tutorial

This notebook demonstrates how to model **time-varying volatility** with a
**Generalized Autoregressive Conditional Heteroskedasticity (GARCH)** model
using the custom numpy-based `GARCHModel` class.

## What is GARCH?

A GARCH(p, q) model captures **volatility clustering** — periods of high
volatility tend to follow other high-volatility periods.

$$\sigma_t^2 = \omega + \sum_{i=1}^{q} \alpha_i \varepsilon_{t-i}^2 + \sum_{j=1}^{p} \beta_j \sigma_{t-j}^2$$

where $\varepsilon_t = \sigma_t z_t$ and $z_t \sim N(0,1)$.

## Dataset

We use Indian stock daily price data from the repository's `sample_data` module.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'testing'))

import numpy as np
import matplotlib.pyplot as plt

from timeseries import (moving_average, autocorrelation,
                        exponential_smoothing, GARCHModel)
from sample_data import RELIANCE_DAILY

## 1. Compute returns and visualise

In [ ]:
prices = np.array(RELIANCE_DAILY)
returns = ((prices[1:] - prices[:-1]) / prices[:-1] * 100).tolist()

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
axes[0].plot(returns)
axes[0].set_title('Reliance Daily Returns (%)')
axes[1].plot([r ** 2 for r in returns])
axes[1].set_title('Squared Returns (volatility proxy)')
plt.tight_layout()
plt.show()

## 2. Explore with library helpers

In [ ]:
# Moving average of absolute returns as a volatility proxy
abs_ret = [abs(r) for r in returns]
ma5 = moving_average(abs_ret, 5)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(abs_ret, alpha=0.4, label='|return|')
ax.plot(ma5, label='5-day MA of |return|', linewidth=2)
ax.set_title('Volatility Proxy — Moving Average of Absolute Returns')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Exponential smoothing of squared returns
sq_ret = [r ** 2 for r in returns]
ema_vol = exponential_smoothing(sq_ret, alpha=0.3)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(sq_ret, alpha=0.3, label='Squared returns')
ax.plot(ema_vol, label='EMA(0.3) of squared returns', linewidth=2)
ax.set_title('Exponential Smoothing of Squared Returns')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Autocorrelation of squared returns
lags = range(1, 10)
acf_sq = [autocorrelation(sq_ret, lag) for lag in lags]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(lags, acf_sq)
ax.set_xlabel('Lag')
ax.set_ylabel('ACF')
ax.set_title('ACF of Squared Returns')
plt.tight_layout()
plt.show()

## 3. Fit a GARCH(1,1) model (numpy-based)

In [ ]:
model = GARCHModel(p=1, q=1)
model.fit(returns)

print('GARCH(1,1) Fitted Parameters')
print(f'  omega : {model.omega:.6f}')
print(f'  alpha : {np.round(model.alpha, 6)}')
print(f'  beta  : {np.round(model.beta, 6)}')

## 4. Conditional volatility

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(returns, alpha=0.4, label='Returns')
ax.plot(model.conditional_volatility, color='red', linewidth=1.5,
        label='GARCH(1,1) Cond. Vol')
ax.plot(-model.conditional_volatility, color='red', linewidth=1.5)
ax.set_title('Returns with GARCH(1,1) Conditional Volatility Bands')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Volatility forecasting

In [ ]:
fc = model.forecast(steps=5)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, 6), fc['volatility'], marker='o')
ax.set_xlabel('Days Ahead')
ax.set_ylabel('Forecasted Volatility (%)')
ax.set_title('GARCH(1,1) — 5-Day Volatility Forecast')
plt.tight_layout()
plt.show()

print('Forecasted volatility:', [round(v, 4) for v in fc['volatility']])

## 6. Standardised residual diagnostics

In [ ]:
eps = np.array(returns) - np.mean(returns)
std_resid = eps / model.conditional_volatility

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(std_resid, bins=8, density=True, alpha=0.7)
axes[0].set_title('Histogram of Standardised Residuals')

acf_resid_sq = [autocorrelation((std_resid ** 2).tolist(), lag) for lag in range(1, 8)]
axes[1].bar(range(1, 8), acf_resid_sq)
axes[1].set_title('ACF of Squared Standardised Residuals')
axes[1].set_xlabel('Lag')
plt.tight_layout()
plt.show()

## Summary

| Step | Tool / Function |
|---|---|
| Volatility proxy | `timeseries.moving_average`, `timeseries.exponential_smoothing` |
| Volatility-clustering evidence | `timeseries.autocorrelation` on squared returns |
| GARCH fitting | `timeseries.GARCHModel` (numpy) |
| Residual diagnostics | `timeseries.autocorrelation` on standardised residuals |